# Assignment 4 - Document Similarity & Topic Modelling

## Part 1 - Document Similarity

For the first part of this assignment, you will complete the functions `doc_to_synsets` and `similarity_score` which will be used by `document_path_similarity` to find the path similarity between two documents.

The following functions are provided:
* **`convert_tag:`** converts the tag given by `nltk.pos_tag` to a tag used by `wordnet.synsets`. You will need to use this function in `doc_to_synsets`.
* **`document_path_similarity:`** computes the symmetrical path similarity between two documents by finding the synsets in each document using `doc_to_synsets`, then computing similarities using `similarity_score`.

You will need to finish writing the following functions:
* **`doc_to_synsets:`** returns a list of synsets in document. This function should first tokenize and part of speech tag the document using `nltk.word_tokenize` and `nltk.pos_tag`. Then it should find each tokens corresponding synset using `wn.synsets(token, wordnet_tag)`. The first synset match should be used. If there is no match, that token is skipped.
* **`similarity_score:`** returns the normalized similarity score of a list of synsets (s1) onto a second list of synsets (s2). For each synset in s1, find the synset in s2 with the largest similarity value. Sum all of the largest similarity values together and normalize this value by dividing it by the number of largest similarity values found. Be careful with data types, which should be floats. Missing values should be ignored.

Once doc_to_synsets and similarity_score have been completed, submit to the autograder which will run a test to check that these functions are running correctly.

*Do not modify the functions `convert_tag` and `document_path_similarity`.*

In [141]:
%%capture
import numpy as np
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('wordnet')
nltk.download('omw-1.4')
from nltk.corpus import wordnet as wn
import pandas as pd
nltk.data.path.append("assets/")

def convert_tag(tag):
    """Convert the tag given by nltk.pos_tag to the tag used by wordnet.synsets"""
    
    tag_dict = {'N': 'n', 'J': 'a', 'R': 'r', 'V': 'v'}
    try:
        return tag_dict[tag[0]]
    except KeyError:
        return None

In [148]:
import nltk
import string
from nltk.tokenize import word_tokenize

def doc_to_synsets(doc):
    """
    Returns a list of synsets in document.

    Tokenizes and tags the words in the document doc.
    Then finds the first synset for each word/tag combination.
    If a synset is not found for that combination it is skipped.

    Args:
        doc: string to be converted

    Returns:
        list of synsets

    Example:
        doc_to_synsets('Fish are friends.')
        Out: [Synset('fish.n.01'), Synset('be.v.01'), Synset('friend.n.01')]
    """
    text = word_tokenize(doc)
    tags = nltk.pos_tag(text)
    # convert tag into one that is perusable in wn 
    # also remove all punctuation
    converted_text = [(word, convert_tag(tag)) for word, tag in tags] 
    
    # initialize empty synsets list 
    synset = []
    for word, tag in converted_text: 
        # run through each pair of word and tags in converted_text
        # pick out the first synset match 
        if wn.synsets(word, pos=tag) != []: # ignore if no synsets matched
            synset.append(wn.synsets(word, pos=tag)[0])     
    
    # YOUR CODE HERE
    #raise NotImplementedError()
    return synset # Your Answer Here


def similarity_score(s1, s2):
    """
    Calculate the normalized similarity score of s1 onto s2

    For each synset in s1, finds the synset in s2 with the largest similarity value.
    Sum of all of the largest similarity values and normalize this value by dividing it by the
    number of largest similarity values found.

    Args:
        s1, s2: list of synsets from doc_to_synsets

    Returns:
        normalized similarity score of s1 onto s2

    Example:
        synsets1 = doc_to_synsets('I like cats')
        synsets2 = doc_to_synsets('I like dogs')
        similarity_score(synsets1, synsets2)
        Out: 0.7333333333333333
    """
    max_score_list = [] 
    
    for synset1 in s1: # loop through each synset in s1 first 
        max_score = 0  # set similarity = 0 first and update
        for synset2 in s2: # go through all synsets in s2 next 
            similarity = synset1.path_similarity(synset2)
            if similarity > max_score: # if similarity is greater than max_score, update max_score
                max_score = similarity 
        max_score_list.append(max_score) # after all s2 elements, update final max_score
    
    # initialise total score
    total = 0 
    for score in max_score_list:
        total += score
        
    # normalise total
    normalised_score = total / len(max_score_list)
            
    
    # YOUR CODE HERE
    #raise NotImplementedError()
    return normalised_score # Your Answer Here

In [143]:
wn.synsets?

Signature: wn.synsets(lemma, pos=None, lang='eng', check_exceptions=True)
Docstring:
Load all synsets with a given lemma and part of speech tag.
If no pos is specified, all synsets for all parts of speech
will be loaded.
If lang is specified, all the synsets associated with the lemma name
of that language will be returned.
File:      /opt/conda/lib/python3.9/site-packages/nltk/corpus/reader/wordnet.py
Type:      method


In [144]:
doc_to_synsets('Fish are friends.')

[Synset('fish.n.01'), Synset('be.v.01'), Synset('friend.n.01')]

In [145]:
doc_to_synsets('Fish are friends.')[0].path_similarity(doc_to_synsets('Fish are friends.')[1])

0.07692307692307693

In [146]:
similarity_score(doc_to_synsets('I like cats'), doc_to_synsets('I like dogs'))

0.7333333333333334

In [147]:
def document_path_similarity(doc1, doc2):
    """Finds the symmetrical similarity between doc1 and doc2"""

    synsets1 = doc_to_synsets(doc1)
    synsets2 = doc_to_synsets(doc2)

    return (similarity_score(synsets1, synsets2) + similarity_score(synsets2, synsets1)) / 2

`paraphrases` is a DataFrame which contains the following columns: `Quality`, `D1`, and `D2`.

`Quality` is an indicator variable which indicates if the two documents `D1` and `D2` are paraphrases of one another (1 for paraphrase, 0 for not paraphrase).

In [149]:
# Use this dataframe for questions most_similar_docs and label_accuracy
paraphrases = pd.read_csv('assets/paraphrases.csv')
paraphrases.head()

,Quality,D1,D2
0,1,"Ms Stewart, the chief executive, was not expec...","Ms Stewart, 61, its chief executive officer an..."
1,1,After more than two years' detention under the...,After more than two years in detention by the ...
2,1,"""It still remains to be seen whether the reven...","""It remains to be seen whether the revenue rec..."
3,0,"And it's going to be a wild ride,"" said Allan ...","Now the rest is just mechanical,"" said Allan H..."
4,1,The cards are issued by Mexico's consulates to...,The card is issued by Mexico's consulates to i...


___

### most_similar_docs

Using `document_path_similarity`, find the pair of documents in paraphrases which has the maximum similarity score.

*This function should return a tuple `(D1, D2, similarity_score)`*

In [152]:
paraphrases['D1'][0]

'Ms Stewart, the chief executive, was not expected to attend.'

In [154]:
def most_similar_docs():
    
    # initialise index count and tuples 
    final = (None, None, None)
    similarity = 0 # initiliase smilarity at first 
    i = 0 
    for doc1 in paraphrases['D1']: 
        D1 = doc1 
        D2 = paraphrases['D2'][i]
        similarity_score = document_path_similarity(D1, D2)
        if similarity_score > similarity: 
            similarity = similarity_score # update max_similarity score
            final = (D1, D2, similarity)
        i += 1 # increment to next index  
    
    # YOUR CODE HERE
    # raise NotImplementedError()
    return final # Your Answer Here

most_similar_docs()

('"Indeed, Iran should be put on notice that efforts to try to remake Iraq in their image will be aggressively put down," he said.',
 '"Iran should be on notice that attempts to remake Iraq in Iran\'s image will be aggressively put down," he said.\n',
 0.9590643274853801)

### label_accuracy

Provide labels for the twenty pairs of documents by computing the similarity for each pair using `document_path_similarity`. Let the classifier rule be that if the score is greater than 0.75, label is paraphrase (1), else label is paraphrase (0). Report accuracy of the classifier using scikit-learn's accuracy_score.

*This function should return a float.*

In [167]:
paraphrases['label']

0     0
1     1
2     1
3     0
4     1
5     1
6     0
7     0
8     0
9     0
10    0
11    0
12    0
13    1
14    0
15    1
16    0
17    0
18    0
19    0
Name: label, dtype: object

In [172]:
def label_accuracy():
    from sklearn.metrics import accuracy_score

    # YOUR CODE HERE
    # add new column 'label' which is initially NaN
    paraphrases['label']=np.NaN
    # compute similarity score for each pair 
    i = 0 
    for doc1 in paraphrases['D1']: 
        D1 = doc1 
        D2 = paraphrases['D2'][i]
        similarity_score = document_path_similarity(D1, D2)
        if similarity_score > 0.75: 
            paraphrases['label'][i] = 1 
        else: # similarity score <= 0.75 
            paraphrases['label'][i] = 0 
        i += 1 # increment to next index  
        
    # compute accuracy 
    accuracy = accuracy_score(paraphrases['Quality'], paraphrases['label'])
    
    # raise NotImplementedError()
    return accuracy # Your Answer Here

label_accuracy()

0.7

## Part 2 - Topic Modelling

For the second part of this assignment, you will use Gensim's LDA (Latent Dirichlet Allocation) model to model topics in `newsgroup_data`. You will first need to finish the code in the cell below by using gensim.models.ldamodel.LdaModel constructor to estimate LDA model parameters on the corpus, and save to the variable `ldamodel`. Extract 10 topics using `corpus` and `id_map`, and with `passes=25` and `random_state=34`.

In [173]:
import pickle
import gensim
from sklearn.feature_extraction.text import CountVectorizer

# Load the list of documents
with open('assets/newsgroups', 'rb') as f:
    newsgroup_data = pickle.load(f)

# Use CountVectorizor to find three letter tokens, remove stop_words, 
# remove tokens that don't appear in at least 20 documents,
# remove tokens that appear in more than 20% of the documents
vect = CountVectorizer(min_df=20, max_df=0.2, stop_words='english', 
                       token_pattern='(?u)\\b\\w\\w\\w+\\b')
# Fit and transform
X = vect.fit_transform(newsgroup_data)

# Convert sparse matrix to gensim corpus.
corpus = gensim.matutils.Sparse2Corpus(X, documents_columns=False)

# Mapping from word IDs to words (To be used in LdaModel's id2word parameter)
id_map = dict((v, k) for k, v in vect.vocabulary_.items())


In [181]:
# Use the gensim.models.ldamodel.LdaModel constructor to estimate 
# LDA model parameters on the corpus, and save to the variable `ldamodel`
import gensim
ldamodel = gensim.models.ldamodel.LdaModel(corpus, num_topics=10,  id2word = id_map, passes=25, random_state=34)
# YOUR CODE HERE
#raise NotImplementedError()

In [183]:
gensim.models.ldamodel.LdaModel?

Init signature:
gensim.models.ldamodel.LdaModel(
    corpus=None,
    num_topics=100,
    id2word=None,
    distributed=False,
    chunksize=2000,
    passes=1,
    update_every=1,
    alpha='symmetric',
    eta=None,
    decay=0.5,
    offset=1.0,
    eval_every=10,
    iterations=50,
    gamma_threshold=0.001,
    minimum_probability=0.01,
    random_state=None,
    ns_conf=None,
    minimum_phi_value=0.01,
    per_word_topics=False,
    callbacks=None,
    dtype=<class 'numpy.float32'>,
)
Docstring:     
Train and use Online Latent Dirichlet Allocation model as presented in
`'Online Learning for LDA' by Hoffman et al.`_

Examples
-------
Initialize a model using a Gensim corpus

.. sourcecode:: pycon

    >>> from gensim.test.utils import common_corpus
    >>>
    >>> lda = LdaModel(common_corpus, num_topics=10)

You can then infer topic distributions on new, unseen documents.

.. sourcecode:: pycon

    >>> doc_bow = [(1, 0.3), (2, 0.1), (0, 0.09)]
    >>> doc_lda = lda[doc_bow]

T

### lda_topics

Using `ldamodel`, find a list of the 10 topics and the most significant 10 words in each topic. This should be structured as a list of 10 tuples where each tuple takes on the form:

`(9, '0.068*"space" + 0.036*"nasa" + 0.021*"science" + 0.020*"edu" + 0.019*"data" + 0.017*"shuttle" + 0.015*"launch" + 0.015*"available" + 0.014*"center" + 0.013*"information"')`

for example.

*This function should return a list of tuples.*

In [188]:
def lda_topics():
    
    ldamodel.print_topics()
    # YOUR CODE HERE
    #raise NotImplementedError()
    return ldamodel.print_topics() # Your Answer Here

In [189]:
ldamodel.print_topics()

[(0,
  '0.056*"edu" + 0.043*"com" + 0.033*"thanks" + 0.022*"mail" + 0.021*"know" + 0.020*"does" + 0.014*"info" + 0.012*"monitor" + 0.010*"looking" + 0.010*"don"'),
 (1,
  '0.024*"ground" + 0.018*"current" + 0.018*"just" + 0.013*"want" + 0.013*"use" + 0.011*"using" + 0.011*"used" + 0.010*"power" + 0.010*"speed" + 0.010*"output"'),
 (2,
  '0.061*"drive" + 0.042*"disk" + 0.033*"scsi" + 0.030*"drives" + 0.028*"hard" + 0.028*"controller" + 0.027*"card" + 0.020*"rom" + 0.018*"floppy" + 0.017*"bus"'),
 (3,
  '0.023*"time" + 0.015*"atheism" + 0.014*"list" + 0.013*"left" + 0.012*"alt" + 0.012*"faq" + 0.012*"probably" + 0.011*"know" + 0.011*"send" + 0.010*"months"'),
 (4,
  '0.025*"car" + 0.016*"just" + 0.014*"don" + 0.014*"bike" + 0.012*"good" + 0.011*"new" + 0.011*"think" + 0.010*"year" + 0.010*"cars" + 0.010*"time"'),
 (5,
  '0.030*"game" + 0.027*"team" + 0.023*"year" + 0.017*"games" + 0.016*"play" + 0.012*"season" + 0.012*"players" + 0.012*"win" + 0.011*"hockey" + 0.011*"good"'),
 (6,
  '0.0

In [207]:
# pip install pyldavis

In [210]:
# visualise the topics 
import pyLDAvis
pyLDAvis.enable_notebook() 
vis = pyLDAvis.gensim.prepare(ldamodel, corpus, id2word)
vis

AttributeError: module 'pyLDAvis' has no attribute 'gensim'

### topic_distribution

For the new document `new_doc`, find the topic distribution. Remember to use vect.transform on the the new doc, and Sparse2Corpus to convert the sparse matrix to gensim corpus.

*This function should return a list of tuples, where each tuple is `(#topic, probability)`*

In [211]:
new_doc = ["\n\nIt's my understanding that the freezing will start to occur because \
of the\ngrowing distance of Pluto and Charon from the Sun, due to it's\nelliptical orbit. \
It is not due to shadowing effects. \n\n\nPluto can shadow Charon, and vice-versa.\n\nGeorge \
Krumins\n-- "]

In [218]:
def topic_distribution():
    
    # Use CountVectorizor to find three letter tokens, remove stop_words, 
    # remove tokens that don't appear in at least 20 documents,
    # remove tokens that appear in more than 20% of the documents
    # vect = CountVectorizer(min_df=20, max_df=0.2, stop_words='english', 
                           # token_pattern='(?u)\\b\\w\\w\\w+\\b')
    # Fit and transform
    X = vect.transform(new_doc)

    # Convert sparse matrix to gensim corpus.
    corpus = gensim.matutils.Sparse2Corpus(X, documents_columns=False)

    # Mapping from word IDs to words (To be used in LdaModel's id2word parameter)
    # id_map = dict((v, k) for k, v in vect.vocabulary_.items())
    
    topic_distri = ldamodel.get_document_topics(corpus) 
    topic_dis_list = [(topic_id, prob) for topic_id, prob in topic_distri[0]]
    
    # YOUR CODE HERE
    # raise NotImplementedError()
    return topic_dis_list # Your Answer Here

In [219]:
topic_distribution()

[(0, 0.020003106),
 (1, 0.020003323),
 (2, 0.02000128),
 (3, 0.49677762),
 (4, 0.020004036),
 (5, 0.020004127),
 (6, 0.020002969),
 (7, 0.020002643),
 (8, 0.020003127),
 (9, 0.34319773)]

In [223]:
ldamodel.get_document_topics(corpus)

### topic_names

From the list of the following given topics, assign topic names to the topics you found. If none of these names best matches the topics you found, create a new 1-3 word "title" for the topic.

Topics: Health, Science, Automobiles, Politics, Government, Travel, Computers & IT, Sports, Business, Society & Lifestyle, Religion, Education.

*This function should return a list of 10 strings.*

In [225]:
ldamodel.print_topics()
topic_list = ["Education", "Travel", "Automobiles", "Reigion", "Automobiles", "Sports", "Science", "Religion", "Computers & IT", "Science"] 

In [226]:
def topic_names():
    
    # YOUR CODE HERE
    topic_list = ["Education", "Travel", "Automobiles", "Reigion", "Automobiles", "Sports", "Science", "Religion", "Computers & IT", "Science"] 
    #raise NotImplementedError()
    return topic_list # Your Answer Here